# Step 3 — 태그별 조회수 효과 + 유의성

`step1_tags_parsed.csv`(태그) + `step2_top_tags.csv`(검정 대상) 로드.
각 (카테고리, 태그)에 대해 같은 카테고리 내 **보유 vs 미보유** log_views 비교:

- median 차이(diff, 자연로그 단위 → 배수효과 exp(diff)) + 양측 Mann-Whitney U
- rank-biserial r (보조 효과크기; 양수 = 보유군이 높음)
- 전체 검정에 BH-FDR 보정(q, sig_fdr) 병기
- 산출: `step3_tag_effect_top3.csv`

In [6]:
import pandas as pd
import numpy as np
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

In [7]:
d = pd.read_csv('step1_tags_parsed.csv')
d['tag_set'] = d['tags_joined'].fillna('').apply(lambda s: set(s.split('|')) if s else set())
top = pd.read_csv('step2_top_tags.csv')
TARGET = ['Gaming', 'Music', 'Film & Animation']
print('videos:', d.shape, '| (cat,tag) pairs:', len(top))

videos: (6249, 5) | (cat,tag) pairs: 45


In [8]:
rows = []
for _, row in top.iterrows():
    cat, tag = row['category'], row['tag']
    sub = d[d['category'] == cat]
    has = sub['tag_set'].apply(lambda s: tag in s)
    a = sub.loc[has, 'log_views'].values   # 보유
    b = sub.loc[~has, 'log_views'].values  # 미보유
    n_with, n_without = len(a), len(b)
    if n_with < 5 or n_without < 1:
        continue
    U, p = mannwhitneyu(a, b, alternative='two-sided')
    rbc = 2.0 * U / (n_with * n_without) - 1   # 양수 = 보유군이 높음
    rows.append({'category': cat, 'tag': tag, 'n_with': n_with, 'n_without': n_without,
                 'median_with': round(float(np.median(a)), 4), 'median_without': round(float(np.median(b)), 4),
                 'diff': round(float(np.median(a) - np.median(b)), 4),
                 'U': float(U), 'p': float(p), 'rank_biserial': round(rbc, 4)})

res = pd.DataFrame(rows)
res['sig'] = np.where(res['p'] < 0.05, '*', '')
rej, q, _, _ = multipletests(res['p'], alpha=0.05, method='fdr_bh')
res['q'] = q.round(4)
res['sig_fdr'] = np.where(rej, '*', '')
print('computed:', res.shape)

computed: (45, 13)


In [9]:
cols = ['category', 'tag', 'n_with', 'n_without', 'median_with', 'median_without', 'diff', 'U', 'p', 'sig', 'q', 'sig_fdr', 'rank_biserial']
res = res[cols]
res.to_csv('step3_tag_effect_top3.csv', index=False)
print('saved: step3_tag_effect_top3.csv', res.shape)

saved: step3_tag_effect_top3.csv (45, 13)


In [10]:
print('카테고리별 유의 태그 수 (raw* / fdr*):')
for cat in TARGET:
    s = res[res['category'] == cat]
    print('  {}: raw {} / fdr {}  (검정 {}개)'.format(cat, int((s['sig']=='*').sum()), int((s['sig_fdr']=='*').sum()), len(s)))
print()

show = ['tag', 'n_with', 'diff', 'p', 'sig', 'sig_fdr', 'rank_biserial']
for cat in TARGET:
    s = res[res['category'] == cat].sort_values('diff', ascending=False)
    print('===', cat, '(diff 내림차순) ===')
    print(s[show].to_string(index=False))
    print()

카테고리별 유의 태그 수 (raw* / fdr*):
  Gaming: raw 1 / fdr 0  (검정 15개)
  Music: raw 6 / fdr 4  (검정 15개)
  Film & Animation: raw 8 / fdr 3  (검정 15개)

=== Gaming (diff 내림차순) ===
            tag  n_with    diff        p sig sig_fdr  rank_biserial
        e3 2018       8  0.5199 0.131550                     0.3253
       xbox one       9  0.4268 0.976227                     0.0072
           game      17  0.0774 0.859512                    -0.0280
       gameplay      24 -0.3809 0.262277                    -0.1526
         action      17 -0.4398 0.464980                    -0.1134
            rpg      15 -0.4398 0.684840                    -0.0667
            fun      14 -0.4815 0.565332                    -0.0969
       nintendo      22 -0.5019 0.237746                    -0.1657
           play      15 -0.6853 0.482734                    -0.1147
nintendo switch      18 -0.7412 0.280953                    -0.1633
     video game      17 -1.0018 0.401137                    -0.1303
           kids 